# 0. Problem
## 1251. Average Selling Price — Easy
For each product, calculate the units-weighted average selling price across its valid price periods. Products with no units sold return `0.00`.

Official: https://leetcode.com/problems/average-selling-price/

# 1. Setup

In [ ]:
import pandas as pd
prices_rows=[(1,"2019-02-17","2019-02-28",5),(1,"2019-03-01","2019-03-22",20),(2,"2019-02-01","2019-02-20",15),(2,"2019-02-21","2019-03-31",30),(3,"2019-01-01","2019-12-31",12)]
units_rows=[(1,"2019-02-25",100),(1,"2019-03-01",15),(2,"2019-02-10",200),(2,"2019-03-22",30)]
prices_pd=pd.DataFrame(prices_rows,columns=["product_id","start_date","end_date","price"])
units_pd=pd.DataFrame(units_rows,columns=["product_id","purchase_date","units"])
prices_pd[["start_date","end_date"]]=prices_pd[["start_date","end_date"]].apply(pd.to_datetime)
units_pd["purchase_date"]=pd.to_datetime(units_pd["purchase_date"])
prices_pd, units_pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark=SparkSession.builder.getOrCreate()
prices_spark=(spark.createDataFrame(prices_rows,["product_id","start_date","end_date","price"]).withColumn("start_date",F.to_date("start_date")).withColumn("end_date",F.to_date("end_date")))
units_spark=spark.createDataFrame(units_rows,["product_id","purchase_date","units"]).withColumn("purchase_date",F.to_date("purchase_date"))
prices_spark.createOrReplaceTempView("Prices")
units_spark.createOrReplaceTempView("UnitsSold")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""
SELECT p.product_id,
       COALESCE(ROUND(SUM(p.price*u.units)/SUM(u.units),2),0.00) AS average_price
FROM Prices p
LEFT JOIN UnitsSold u
  ON p.product_id=u.product_id
 AND u.purchase_date BETWEEN p.start_date AND p.end_date
GROUP BY p.product_id
ORDER BY p.product_id
""")
sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
joined_pd=prices_pd.merge(units_pd,on="product_id",how="left")
matched_pd=joined_pd.loc[joined_pd["purchase_date"].between(joined_pd["start_date"],joined_pd["end_date"])].copy()
matched_pd["weighted_amount"]=matched_pd["price"]*matched_pd["units"]
weighted_pd=(matched_pd.groupby("product_id",as_index=False).agg(weighted_amount=("weighted_amount","sum"),units=("units","sum")))
weighted_pd["average_price"]=(weighted_pd["weighted_amount"]/weighted_pd["units"]).round(2)
result_pd=(prices_pd[["product_id"]].drop_duplicates().merge(weighted_pd[["product_id","average_price"]],on="product_id",how="left").fillna({"average_price":0.0}).sort_values("product_id").reset_index(drop=True))
result_pd

# 4. PySpark Solution

In [ ]:
j=prices_spark.alias("p").join(units_spark.alias("u"),(F.col("p.product_id")==F.col("u.product_id"))&F.col("u.purchase_date").between(F.col("p.start_date"),F.col("p.end_date")),"left")
result_spark=(j.groupBy(F.col("p.product_id").alias("product_id")).agg(F.coalesce(F.round(F.sum(F.col("p.price")*F.col("u.units"))/F.sum(F.col("u.units")),2),F.lit(0.0)).alias("average_price")).orderBy("product_id"))
result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| date-range join | `BETWEEN` in join | merge + `.between()` | join condition + `.between()` |
| weighted average | `SUM(price*units)/SUM(units)` | weighted amount / units | same formula |
| no sales default | `COALESCE(...,0)` | left merge + `.fillna(0)` | `F.coalesce()` |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Prices, UnitsSold

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: prices_pd, units_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: prices_spark, units_spark